## Variant region list:
- load the file of all sequences/oligo names (80215)
- make a list of these with a column for the name, sequence, if variant, if ref, if region, if control, association with gene?


In [16]:
# imports 
import pandas as pd 
from Bio import SeqIO
# use the config file in yaml format
import yaml
import os

# read config
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

In [14]:

# load the data
design_fasta = '/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa'

# read the fasta file with the sequences and prepare a tsv with header and sequence using biopython

records = list(SeqIO.parse(design_fasta, "fasta"))
design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

design_df['header'] = header
design_df['sequence'] = sequence

# label is the string in front of the first ":"
design_df['label'] = design_df['header'].str.split(':').str[0]
design_df


    

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


In [15]:
# investigate the variants
for i in range(40, 48):
    print(design_df['header'].tolist()[i])


cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779571_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779574_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779580_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779583_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779643_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779653_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779655_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779714_fwd_tile1-1


In [41]:
# filter for the rows with "cardiac_neuro_cava_random" in label column
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']
# do all of these rows have 2 pipes in the header?
cardiac_neuro_cava_random['header'].str.count('\|').value_counts()

# header
# 5     45082
# 2     27141
# 8       596
# 7       471
# 4       341
# 10      30


header
5     45082
2     27141
8       596
7       471
4       341
10      309
Name: count, dtype: int64

#### 2 pipe example
- 18340 REF
- 27141 all


In [60]:
# show me an example with threshold pipes in the header
num_pipes = 2
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1'

# does a row with 2 pipes in the header have ALT in the header?
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].str.contains('ALT_').value_counts() # no alt there
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].str.contains('REF_').value_counts() # no ref there # 18340 REF

# investigate the rows with 2 pipes in the header and ALT in the header
card_cava_2_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]
print(card_cava_2_pipes.shape) # (27141, 3)
# card_cava_2_pipes['header'].str.contains('ALT')]['header'].tolist()[0]
card_cava_2_pipes[card_cava_2_pipes['header'].str.contains('REF_')]['header'].tolist()[10] # 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

(27141, 3)


'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

#### 4 pipe example (what group is this, mohan?)
- 242 REF / 99 not

In [79]:
# show me an example with threshold pipes in the header
num_pipes = 4
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] # 341 rows
print(len(card_cava_4_pipes)) # 341
# do all these rows have ALT in the header?
card_cava_4_pipes['header'].str.contains('ALT_').value_counts() # no, no alt 
card_cava_4_pipes['header'].str.contains('REF_').value_counts() # True, 242 

# check the header of something containing REF_
card_cava_4_pipes[card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1'

# check the header of something not containing REF_
card_cava_4_pipes[~card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

341


'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

#### 5 pipe examples are all alt / variants (45082)
- 45082 alt

In [82]:
num_pipes = 5
# show me an example with 5 pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# idx 0: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C

# investigate and find pattern
card_cava_5_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_5_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # yes => all with 5 pipes have ALT_ in name (45082)
# card_cava_5_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() #

# # do all of these rows end with the regex /-[A-Z]*-[A-Z]*/
# card_cava_5_pipes['header'].str.extract(r'(-[A-Z]*-[A-Z]*)$')[0].value_counts() # yes => all with 5 pipes have ALT_ in name


header
False    45082
Name: count, dtype: int64

#### 7 pipe examples (471) 
- all alt 471

In [70]:
num_pipes = 7
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'

# # investigate and find pattern
# card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 471
# # card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref


'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'

#### 8 pipe example 596
- 596 alt

In [69]:
num_pipes = 8
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

# investigate and find pattern
card_cava_8_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 596
# card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

#### 10 pipes in example 309
- 309 alt

In [75]:
num_pipes = 10
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'

# investigate and find pattern
card_cava_10_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 309
# card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

header
False    309
Name: count, dtype: int64

### Prepare a tsv of all labels and get the number of sequences

In [23]:
# write sequences with same label in one file
# get list of all labels in dataframe
labels = design_df['label'].unique().tolist()
group_list_output_dir = config['general']['group_lists_directory']

for label_of_interest in labels: 
    # get the rows of the dataframe with the label of interest
    label_of_interest_df = design_df[design_df['label'] == label_of_interest]
    # write it to directory as tsv file
    label_of_interest_df.to_csv(os.path.join(group_list_output_dir, label_of_interest + f'_{len(label_of_interest_df)}.tsv'), sep='\t', index=False)


In [24]:
# get number of all rows with a label starting with "C_"
C_labels = design_df[design_df['label'].str.startswith('C_')]
C_labels

,header,sequence,label
74811,C_positive_heart_CAD:REF_rs17114036,AGGACCGGATCAACTAGGAAGCAGGTCATAATTAGTGATAGTCATT...,C_positive_heart_CAD
74812,C_positive_heart_CAD:REF_rs72664324,AGGACCGGATCAACTTCCTCTGCTGAACCCACAGCAATGGCAGCCG...,C_positive_heart_CAD
74813,C_positive_heart_CAD:REF_rs12740374,AGGACCGGATCAACTTGACCCAAAAGTGCTTCATTTTTCGTGCCCG...,C_positive_heart_CAD
74814,C_positive_heart_CAD:REF_rs4450010,AGGACCGGATCAACTTGAGGTCCAAGGATGTGAGAGTGACCACAGT...,C_positive_heart_CAD
74815,C_positive_heart_CAD:REF_rs34091558,AGGACCGGATCAACTCTTCTCGGCCAATGAAGGGTCAACTCCATTG...,C_positive_heart_CAD
...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,C_SLEA
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,C_SLEA
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,C_SLEA
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,C_SLEA
